# Synthetic case two: saved SoS posterior

Run All reconstructs the accepted primary baseline from its supplied samples. **No optimization, candidate selection or TMCMC is called.**

The shared `sumsigmoid.model.Model` uses the total-population denominator. Dates and ML-selected signs are fixed; their selection uncertainty is excluded. Completion of tempering does not establish posterior convergence.

In [ ]:
from pathlib import Path
import sys, json
import numpy as np

# Paths are relative to this package; Jupyter and VS Code cwd are supported.
def package_root():
    # Support Jupyter's notebook cwd and VS Code's repository/workspace cwd.
    for parent in (Path.cwd(), *Path.cwd().parents):
        for candidate in (parent, parent / "automatic_tuning"):
            if (candidate / "sumsigmoid" / "model.py").is_file() and (candidate / "saved_baselines").is_dir():
                return candidate.resolve()
    raise RuntimeError("Open the notebook inside the supplied package or repository")
PACKAGE_ROOT = package_root()
sys.path.insert(0, str(PACKAGE_ROOT))
from sumsigmoid.posterior import regenerate, load_inputs, generate

CASE_DIR = PACKAGE_ROOT / "saved_baselines" / "case2"
RUN_CONFIGURATION = CASE_DIR / "run_configuration.json"
SAMPLES = CASE_DIR / "posterior_samples.csv"
DATA_FILE = CASE_DIR / "observed_daily.csv"
TRUTH_FILE = CASE_DIR / "generating_curves.npz"
REFERENCE = CASE_DIR / "reference"
OUTPUT = PACKAGE_ROOT / "regenerated" / "case2"
DRAW_ID = None  # None = every saved sample; integer 0..3999 = one selected row
print("Inputs:", CASE_DIR)
print("Shared model:", PACKAGE_ROOT / "sumsigmoid" / "model.py")

## Reconstruct continuously

Days 0–159 are inference observations; the forecast begins at day 160 and continues to 199.9 at Δt=0.1. Compartments are never reset at the cutoff and held-out data never update the state. E₀/I₀ are read from every sample. A selected sample is checked against its matching ensemble row.

Synthetic generating truth, raw observations and model predictions remain separate. The original case-one post-cutoff zeros are placeholders, not observations; use `is_observed`. Toronto contains real active counts and no latent truth.

In [ ]:
result = regenerate(RUN_CONFIGURATION, SAMPLES, DATA_FILE, TRUTH_FILE, OUTPUT,
                    draw_id=DRAW_ID, reference=REFERENCE)
print(json.dumps(result, indent=2))

## Outputs and scoring

`latent_predictive_native.npz` contains E, I and beta with axes **(draw, time)** plus separate `I_obs` Gaussian observation-predictive draws. `daily.npz` uses integer days 0–199 and, in all-sample mode, includes 23 analytic Gaussian-mixture predictive quantiles. All 4,000 equal-weight particles, including duplicates, are retained.

Mean trajectories and empirical linear 2.5/97.5% latent quantiles match the accepted references. Observation-predictive limits include the 10% Gaussian noise; these differ from latent intervals and from empirical quantiles of one noise realization per particle. Fit scoring uses days 1–159 and forecast scoring days 160–199, while likelihood inference includes day 0. Synthetic scores target generating latent I; Toronto scores target observed active I.

The eleven-interval WIS uses central coverages 10,20,…,90,95,98%, median weight 0.5, interval weights α/2 and divisor 11.5, followed by a time mean. It is our accepted implementation, **not a verified copy of Brandon’s unavailable scoring function**. Selected-sample mode does not report posterior scores. These are deterministic checks of existing samples, not new inference.

In [ ]:
native = np.load(OUTPUT / "latent_predictive_native.npz")
daily = np.load(OUTPUT / "daily.npz")
summary = np.load(OUTPUT / "summaries.npz")
print("Draw IDs:", native["draw_id"][:10], "Number of draws:", len(native["draw_id"]))
print("Native I:", native["I"].shape, "Daily I:", daily["I"].shape)
print("Figure:", OUTPUT / "fit_forecast_beta.pdf")
from IPython.display import display, SVG
display(SVG(filename=str(OUTPUT / "fit_forecast_beta.svg")))